# Vaardigheden week 11: patroon of toeval

Dit notebook hoort bij het vaardighedenspoor. Elke dataset erin is verzonnen met een seed, dus je kent de waarheid vooraf en ziet wanneer een methode die vindt en wanneer een methode zich laat misleiden.

Je voert met de hand een permutatietoets uit, leest de p-waarde in één eerlijke zin, ziet waarom een grote steekproef een klein verschil belangrijk laat lijken, toetst een trend, leert wat schudden aanneemt over de tijd, en ziet wat er gebeurt als je veel vergelijkingen probeert en de beste houdt. De formule komt als laatste, als kortere weg voor een toets die je al begrijpt.

**Zo werk je ermee:** voer elke cel op volgorde uit met de afspeelknop of Shift+Enter. Lees de uitvoer voordat je verdergaat.

## 1. De vraag die elke toets stelt

Een boerin test een nieuwe meststof op 24 percelen met hetzelfde gewas. Ze kiest willekeurig 12 percelen voor de meststof en laat de andere 12 zoals ze zijn. Bij de oogst weegt ze de opbrengst van elk perceel in kilo's.

De bemeste percelen brengen gemiddeld meer op. Zo'n verschil kan van de meststof komen, of van het toeval welke percelen hem kregen. Elke toets van een patroon stelt dezelfde vraag: **als de meststof niets deed, hoe vaak zou toeval alleen dan een verschil van deze grootte geven?**

De data hieronder zijn gemaakt met een echt effect van 4 kg, dus je weet dat het juiste antwoord is "de meststof helpt".

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(28)
control = rng.normal(40, 5, 12).round(1)      # 12 percelen zonder meststof, kg
treated = rng.normal(44, 5, 12).round(1)      # 12 percelen met meststof, kg

gap = treated.mean() - control.mean()
print("Gemiddelde opbrengst zonder meststof (kg):", round(control.mean(), 2))
print("Gemiddelde opbrengst met meststof (kg):", round(treated.mean(), 2))
print("Verschil (kg):", round(gap, 2))

## 2. Eén keer schudden

Als de meststof niets deed, zijn de labels "bemest" en "niet bemest" alleen namen die op 24 opbrengsten zijn geplakt. Elke andere manier om er 12 van elk label op te plakken was even waarschijnlijk.

**Schudden** toetst dat idee. Gooi de 24 opbrengsten op één hoop, deel ze willekeurig uit in twee groepen van 12 en bereken het verschil opnieuw. De cel hieronder schudt vijf keer.

In [ ]:
pooled = np.concatenate([treated, control])
shuffler = np.random.default_rng(1)
for i in range(1, 6):
    shuffled = shuffler.permutation(pooled)
    print("schudbeurt", i, "verschil (kg):", round(shuffled[:12].mean() - shuffled[12:].mean(), 2))

## 3. Tienduizend keer schudden

Vijf schudbeurten laten zien dat toeval alleen verschillen van een paar kilo in beide richtingen maakt. Tienduizend schudbeurten laten zien hoe vaak toeval een verschil maakt dat even groot is als het echte.

De **p-waarde** is het aandeel schudbeurten met een verschil dat minstens even groot is als het echte verschil, in beide richtingen. De toets telt beide richtingen, omdat vóór de oogst niemand wist of de meststof zou helpen of schaden.

De functie `shuffle_gaps` doet het werk. Je gebruikt hem in latere delen opnieuw.

In [ ]:
def shuffle_gaps(a, b, n_shuffles, rng):
    """Het verschil tussen twee groepen na elk van n_shuffles willekeurige herverdelingen."""
    pooled = np.concatenate([a, b])
    gaps = np.empty(n_shuffles)
    for i in range(n_shuffles):
        s = rng.permutation(pooled)
        gaps[i] = s[:len(a)].mean() - s[len(a):].mean()
    return gaps

gaps = shuffle_gaps(treated, control, 10000, np.random.default_rng(2))
as_large = int(np.sum(np.abs(gaps) >= abs(gap)))
p_value = as_large / 10000
print("Schudbeurten met een verschil minstens zo groot als het echte:", as_large, "van 10000")
print("p-waarde:", round(p_value, 4))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(gaps, bins=60, color="#999999")
ax.axvline(gap, color="#c0392b", label="echt verschil")
ax.axvline(-gap, color="#c0392b", linestyle="--", label="even groot, andere richting")
ax.set_title("Verschillen uit 10.000 keer schudden van de meststoflabels")
ax.set_xlabel("verschil in gemiddelde opbrengst (kg)")
ax.set_ylabel("schudbeurten")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Eén eerlijke zin

Een p-waarde beantwoordt één vraag en alleen die. De cel hieronder schrijft die vraag als een zin, met jouw getallen ingevuld.

Twee andere lezingen zijn fout, en beide komen vaak voor.

- **De p-waarde is niet de kans dat de meststof niets doet.** De toets neemt aan dat de meststof niets doet en vraagt hoe verrassend de data onder die aanname zouden zijn.
- **De p-waarde is niet de grootte van het effect.** Een kleine p-waarde kan samengaan met een verschil dat te klein is om ertoe te doen, zoals deel 5 laat zien.

Veel vakgebieden noemen een resultaat "statistisch significant" als de p-waarde onder 0,05 ligt. Die grens is een afspraak. Een resultaat van 0,049 en een resultaat van 0,051 dragen bijna hetzelfde bewijs.

In [ ]:
print(f"Als de meststof niets deed, zou een willekeurige verdeling van de percelen in {p_value:.1%} "
      f"van de verdelingen een verschil van minstens {abs(gap):.2f} kg geven.")

## 5. Significant en klein

Een grote steekproef kan een heel klein verschil belangrijk laten lijken. De volgende cel vergelijkt twee zaadrassen op elk 20.000 percelen. De data zijn gemaakt met een echt verschil van 0,3 kg op percelen die ongeveer 40 kg opbrengen.

Als geen enkele schudbeurt het echte verschil haalt, ligt de p-waarde onder één gedeeld door het aantal schudbeurten. Een p-waarde is nooit precies nul, dus schrijf nooit p = 0.

In [ ]:
rng = np.random.default_rng(5)
variety_a = rng.normal(40.0, 5, 20000)
variety_b = rng.normal(40.3, 5, 20000)
big_gap = variety_b.mean() - variety_a.mean()
big_gaps = shuffle_gaps(variety_b, variety_a, 1000, np.random.default_rng(6))
print("Verschil (kg):", round(big_gap, 3))
print("Schudbeurten met een verschil minstens zo groot:", int(np.sum(np.abs(big_gaps) >= abs(big_gap))), "van 1000")
print("Verschil als aandeel van de gemiddelde opbrengst:", f"{big_gap / variety_a.mean():.2%}")
print("Verschil in standaardafwijkingen:", round(big_gap / variety_a.std(), 3))

Het verschil is echt, want de data zijn ermee gemaakt, en het is te klein om voor een boer iets uit te maken. Het getal dat antwoord geeft op "doet het ertoe?" is de **effectgrootte**: het verschil in kilo's, het verschil als aandeel van het gemiddelde, of het verschil in standaardafwijkingen. Zet de effectgrootte naast elke p-waarde.

## 6. Hetzelfde effect bij drie steekproefgroottes

De p-waarde hangt van twee dingen af: hoe groot het effect is en hoeveel data je hebt. De cel hieronder maakt data met hetzelfde echte effect van 2 kg bij drie steekproefgroottes.

In [ ]:
for n in [10, 100, 1000]:
    rng = np.random.default_rng(n)
    a = rng.normal(40, 5, n)
    b = rng.normal(42, 5, n)
    g = b.mean() - a.mean()
    p = np.mean(np.abs(shuffle_gaps(b, a, 2000, np.random.default_rng(7))) >= abs(g))
    print(f"{n} percelen per groep: verschil {g:.2f} kg, p-waarde {p:.4f}")

Het echte effect verandert nooit. De schatting ervan komt dichter bij 2 kg naarmate de steekproef groeit, en de p-waarde daalt. Een grote p-waarde uit een kleine steekproef betekent "deze steekproef kan het effect niet van toeval onderscheiden". Het betekent niet "er is geen effect".

## 7. Een trend in de tijd

Een trend is een patroon in de tijd, en schudden werkt daar ook voor. De verzonnen tabel hieronder bevat 30 jaar bezoekers aan een natuurpark, in duizenden. De helling van een rechte lijn door de punten is de trend, in duizenden bezoekers per jaar. De data zijn gemaakt zonder enige trend.

Om een helling te toetsen, schud je de volgorde van de waarden. Als er geen trend is, was elke volgorde van de waarden even waarschijnlijk als de echte.

In [ ]:
rng = np.random.default_rng(59)
years = np.arange(1996, 2026)
visitors = (50 + rng.normal(0, 4, 30)).round(1)     # duizenden, gemaakt zonder trend
park = pd.DataFrame({"year": years, "visitors_k": visitors})
print(park.tail(5))

We gaven een AI-assistent de laatste vijf rijen en vroegen: *"Stijgt het aantal bezoekers?"* De assistent schreef de cel hieronder. Voer hem uit.

In [ ]:
# Geschreven door een AI-assistent
recent = park.tail(5)
slope = np.polyfit(recent["year"], recent["visitors_k"], 1)[0]
print("Duidelijke stijgende trend: het aantal bezoekers groeit met", round(slope, 2), "duizend per jaar")

De rekensom klopt. De bewering is het probleem, want vijf waarden kunnen door toeval omhoog hellen. Vijf waarden hebben maar 120 mogelijke volgordes, dus de toets kan ze allemaal proberen in plaats van willekeurig te schudden. Daarna loopt dezelfde toets op alle 30 jaar.

In [ ]:
import itertools

def slope_of(values, x):
    return np.polyfit(x, values, 1)[0]

x5 = recent["year"].to_numpy()
orders = np.array([slope_of(np.array(order), x5) for order in itertools.permutations(recent["visitors_k"])])
p_five = np.mean(np.abs(orders) >= abs(slope) - 1e-9)
print("Volgordes van de vijf waarden:", len(orders))
print("p-waarde over vijf jaar:", round(p_five, 3))

all_slope = slope_of(visitors, years)
shuffler = np.random.default_rng(8)
all_slopes = np.array([slope_of(shuffler.permutation(visitors), years) for _ in range(5000)])
p_all = np.mean(np.abs(all_slopes) >= abs(all_slope))
print("Helling over alle 30 jaar (duizend per jaar):", round(all_slope, 3))
print("p-waarde over dertig jaar:", round(p_all, 3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(years, visitors, "o", color="#999999")
ax.plot(x5, np.polyval(np.polyfit(x5, recent["visitors_k"], 1), x5), color="#c0392b", label="lijn door de laatste vijf jaar")
ax.plot(years, np.polyval(np.polyfit(years, visitors, 1), years), color="#2c3e50", label="lijn door alle 30 jaar")
ax.set_title("Parkbezoekers: vijf jaar tegenover dertig")
ax.set_xlabel("jaar")
ax.set_ylabel("bezoekers (duizenden)")
ax.legend()
plt.tight_layout()
plt.show()

Vergelijk de twee p-waarden. Vijf jaar kan toeval niet uitsluiten, en de volledige reeks laat geen trend zien, wat de waarheid is waarmee de data zijn gemaakt. Vraag voordat je een trend aanneemt op hoeveel jaren hij rust, en hoe vaak zoveel waarden in willekeurige volgorde even steil zouden hellen.

## 8. Wat schudden aanneemt

Schudden behandelt elke volgorde van de data als even waarschijnlijk wanneer er geen patroon is. Dat klopt als elke waarde een losse trekking is, zoals willekeurig gekozen percelen. Het klopt niet als de data in de tijd lopen en elke week dicht bij de week ervoor ligt.

De twee verzonnen weekreeksen hieronder hebben geen enkel verband met elkaar. Elke reeks drijft langzaam op en neer, zoals prijzen, rivierstanden en aantallen ziektegevallen vaak doen.

In [ ]:
def drift(rng, n, keep=0.95):
    """Een reeks waarin elke week het grootste deel van de waarde van vorige week houdt en wat ruis toevoegt."""
    x = np.zeros(n)
    noise = rng.normal(0, 1, n)
    for t in range(1, n):
        x[t] = keep * x[t - 1] + noise[t]
    return x

rng = np.random.default_rng(128)
series_a = 100 + 5 * drift(rng, 300)
series_b = 50 + 3 * drift(rng, 300)
r = np.corrcoef(series_a, series_b)[0, 1]
print("Correlatie tussen de twee reeksen zonder verband:", round(r, 3))

fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
ax[0].plot(series_a)
ax[0].set_ylabel("reeks A")
ax[0].set_title("Twee verzonnen reeksen zonder verband")
ax[1].plot(series_b, color="#c0392b")
ax[1].set_ylabel("reeks B")
ax[1].set_xlabel("week")
plt.tight_layout()
plt.show()

Gewoon schudden zet de weken van reeks B in een willekeurige volgorde. Daarmee verdwijnt het langzame drijven, dus de geschudde kopieën lijken in niets op een echte drijvende reeks.

In [ ]:
shuffler = np.random.default_rng(9)
shuffled_r = np.array([np.corrcoef(series_a, shuffler.permutation(series_b))[0, 1] for _ in range(5000)])
p_ordinary = np.mean(np.abs(shuffled_r) >= abs(r))
print("Gewoon schudden: schudbeurten minstens zo sterk:", int(np.sum(np.abs(shuffled_r) >= abs(r))), "van 5000")

Gewoon schudden zegt dat een correlatie van deze sterkte bijna nooit door toeval ontstaat, en dat klopt niet, want de reeksen zijn gemaakt zonder verband. Een **verschuivingstoets** houdt elke reeks in haar eigen volgorde en schuift de ene langs de andere. Elke verschuiving houdt het drijven intact, en alleen de uitlijning tussen de twee reeksen verandert. `np.roll` zet de waarden die aan het eind wegvallen weer vooraan. Verschuivingen korter dan een half jaar (26 weken) vallen weg, zodat geen week tegenover zichzelf of een naaste buur komt te staan.

In [ ]:
shifts = range(26, len(series_b) - 25)
shifted_r = np.array([np.corrcoef(series_a, np.roll(series_b, s))[0, 1] for s in shifts])
p_shift = np.mean(np.abs(shifted_r) >= abs(r))
print("Geprobeerde verschuivingen:", len(shifted_r))
print("p-waarde van de verschuivingstoets:", round(p_shift, 3))

Twee langzaam drijvende reeksen lopen over een paar honderd weken vaak toevallig gelijk op. De verschuivingstoets houdt daar rekening mee en gewoon schudden niet. Vraag voordat je schudt of de volgorde van je data informatie bevat. Als dat zo is, schud dan op een manier die de volgorde bewaart.

## 9. Veel pogingen, één winnaar

Een team noteert 20 gewoonten van 40 verzonnen studenten (uren slaap, minuten lezen enzovoort) naast hun examencijfers. De data zijn zo gemaakt dat geen enkele gewoonte het cijfer beïnvloedt. Het team toetst elke gewoonte en meldt de gewoonte met de kleinste p-waarde.

De eerlijke toets herhaalt de hele zoektocht binnen elke schudbeurt. Hij vraagt hoe vaak de beste van 20 nutteloze gewoonten er even sterk uitziet als de winnaar.

In [ ]:
rng = np.random.default_rng(68)
score = rng.normal(0, 1, 40)
habits = rng.normal(0, 1, (40, 20))       # 20 gewoonten, geen enkele verbonden met het cijfer

def corr_with(y, X):
    """De correlatie van y met elke kolom van X."""
    yc = (y - y.mean()) / y.std()
    Xc = (X - X.mean(axis=0)) / X.std(axis=0)
    return Xc.T @ yc / len(y)

real_r = corr_with(score, habits)
best = int(np.argmax(np.abs(real_r)))
print("Gewoonte met de sterkste correlatie:", best, "met r =", round(real_r[best], 3))

shuffler = np.random.default_rng(10)
one_habit, best_of_all = [], []
for _ in range(5000):
    r_shuffled = corr_with(shuffler.permutation(score), habits)
    one_habit.append(abs(r_shuffled[best]))
    best_of_all.append(np.abs(r_shuffled).max())
p_one = np.mean(np.array(one_habit) >= abs(real_r[best]))
p_search = np.mean(np.array(best_of_all) >= abs(real_r[best]))
print("p-waarde alsof alleen deze gewoonte was getoetst:", round(p_one, 4))
print("p-waarde voor de beste van 20 gewoonten:", round(p_search, 4))
print("Verwachte valse alarmen onder 20 nutteloze gewoonten bij de grens van 0,05:", 20 * 0.05)

Meld hoeveel vergelijkingen je hebt geprobeerd, en corrigeer ervoor. Een resultaat dat alleen overeind blijft als je de andere 19 pogingen verzwijgt, is een resultaat dat toeval kan maken.

## 10. De formule komt daarna

Statistici vonden formules die de uitkomst van het schudden voorspellen zonder te schudden. De bekendste is de t-toets. Die neemt aan dat de twee groepen losse willekeurige steekproeven zijn. Op de meststofdata geeft hij een p-waarde die dicht bij die van het schudden ligt.

Hetzelfde soort formule, gebruikt op de drijvende reeksen uit deel 8, laat zich op dezelfde manier misleiden als gewoon schudden. Een formule is een snelle kortere weg voor schudden, en ze neemt de aannames van het schudden mee.

In [ ]:
from scipy import stats

t_result = stats.ttest_ind(treated, control, equal_var=False)
print("p-waarde van het schudden, meststof:", round(p_value, 4))
print("p-waarde van de t-toets, meststof:", round(t_result.pvalue, 4))
print("p-waarde van de formule, twee drijvende reeksen zonder verband:", f"{stats.pearsonr(series_a, series_b).pvalue:.1e}")
print("p-waarde van de verschuivingstoets, dezelfde reeksen:", round(p_shift, 3))

## 11. Jouw beurt: twee bezorgroutes

Een bezorgbedrijf meet de tijd van 15 ritten op route A en 15 ritten op route B, in minuten. Vraag je AI-assistent om code die een permutatietoets uitvoert op het verschil in gemiddelde tijd (route B min route A), met minstens 10.000 schudbeurten. Sla het verschil op in `gap_route` en de p-waarde in `p_route`, en voer daarna de CHECK-cel uit.

Controleer twee dingen voordat je de code vertrouwt. Schudt de code alle 30 ritten samen? Telt de code verschillen in beide richtingen?

In [ ]:
rng = np.random.default_rng(37)
route_a = rng.normal(52, 8, 15).round(1)
route_b = rng.normal(57, 8, 15).round(1)
print("Gemiddelde route A (minuten):", round(route_a.mean(), 2))
print("Gemiddelde route B (minuten):", round(route_b.mean(), 2))

In [ ]:
# jouw beurt 1: sla het verschil (route B min route A) op in gap_route en de p-waarde in p_route
gap_route = None
p_route = None

In [ ]:
# CHECK 1
assert gap_route is not None and p_route is not None, "gap_route of p_route is nog leeg"
assert abs(gap_route - (route_b.mean() - route_a.mean())) < 1e-6, "het verschil moet route B min route A zijn"
reference = shuffle_gaps(route_b, route_a, 50000, np.random.default_rng(99))
p_reference = np.mean(np.abs(reference) >= abs(gap_route))
assert abs(p_route - p_reference) < 0.02, "de p-waarde klopt niet: telt de code beide richtingen?"
print("CHECK 1 geslaagd: verschil", round(gap_route, 2), "minuten, p-waarde dicht bij", round(p_reference, 3))

## 12. Jouw beurt: wordt de wachttijd langer?

Een verzonnen kliniek noteerde 20 jaar lang de gemiddelde wachttijd per jaar, in minuten. Vraag je AI-assistent om een permutatietoets op de helling. Sla de helling in minuten per jaar op in `wait_slope` en de p-waarde uit minstens 10.000 keer schudden van de wachttijden in `p_wait`. Gebruik gerust `slope_of` uit deel 7.

In [ ]:
rng = np.random.default_rng(13)
wait_years = np.arange(2006, 2026)
wait = (35 + 0.4 * (wait_years - 2006) + rng.normal(0, 4, 20)).round(1)
print(pd.DataFrame({"jaar": wait_years, "wachttijd_minuten": wait}).tail())

In [ ]:
# jouw beurt 2: sla de helling op in wait_slope en de p-waarde in p_wait
wait_slope = None
p_wait = None

In [ ]:
# CHECK 2
assert wait_slope is not None and p_wait is not None, "wait_slope of p_wait is nog leeg"
assert abs(wait_slope - np.polyfit(wait_years, wait, 1)[0]) < 1e-6, "de helling moet komen uit een rechte lijn door alle 20 jaar"
checker = np.random.default_rng(97)
reference = np.array([np.polyfit(wait_years, checker.permutation(wait), 1)[0] for _ in range(20000)])
p_reference = np.mean(np.abs(reference) >= abs(wait_slope))
assert abs(p_wait - p_reference) < 0.02, "de p-waarde klopt niet: schud de wachttijden, houd de jaren vast, tel beide richtingen"
print("CHECK 2 geslaagd: helling", round(wait_slope, 3), "minuten per jaar, p-waarde dicht bij", round(p_reference, 3))

## 13. Zeg het zonder te overdrijven

Iemand vraagt altijd: "Dus het is waar?" Een toets kan dat niet rechtstreeks beantwoorden. Hij zegt hoe vaak toeval alleen zou opleveren wat jij vond, en niets over het waarom. Schrijf elk resultaat twee keer op: één keer voor een specialist en één keer voor iedereen. De cel hieronder doet het voor de meststof.

In [ ]:
print("Voor een specialist:")
print(f"  Een permutatietoets met 10.000 schudbeurten gaf p = {p_value:.3f} voor een verschil van {gap:.1f} kg.")
print("Voor iedereen:")
print(f"  Als de meststof niets deed, zou een verschil van deze grootte opduiken in ongeveer {round(p_value * 100)} van elke 100 willekeurige verdelingen van de percelen.")
print(f"  Toeval is een onwaarschijnlijke verklaring, maar geen onmogelijke. Het verschil was {gap:.1f} kg op percelen die ongeveer {control.mean():.0f} kg opbrengen.")

Schrijf nu beide versies voor je bezorgroutes in een tekstcel hieronder. De versie voor iedereen mag de woorden "p-waarde" en "significant" niet bevatten.